# hsiplot walkthrough on synthetic data

This notebook needs neither the Living Optics SDK nor real data. It shows, in order: quality control, why validation must hold out whole plots, the plot-level tests, and the detectability check. Replace the synthetic table with `load_table("your_spectra.csv")` to use your own data (see the README for the table layout).

In [ ]:
from hsiplot import (
                     PreprocessConfig,
                     detectability,
                     group_difference_test,
                     make_synthetic_table,
                     plot_qc,
                     preprocess_table,
                     run_classification,
)
from hsiplot.validate import random_split_demo

# 12 plots (2 control), 60 spectra each; treated plots have a 50 % higher NIR signal
table = make_synthetic_table(n_plots=12, n_control=2, n_spectra=60, n_bands=48, effect=0.5, plot_sd=0.05, seed=1)
table.plot_table()

## 1. Quality control per plot
A plot far darker or brighter than the median plot, or with many invalid spectra, was measured under a different exposure and is flagged `CHECK`.

In [ ]:
plot_qc(table)

## 2. Cleaning
Invalid values, extreme brightness and rough spectra are removed **within each plot**; the spectra are then smoothed along wavelength. No group labels are used.

In [ ]:
clean = preprocess_table(table, PreprocessConfig.preset("clean")).subsample(60, seed=0)
len(table.df), len(clean.df)

## 3. Why whole-plot validation matters
Same data, but now **without any treatment effect** and with each plot having its own overall level. A random split by spectrum recognises plots it has already seen and scores high; validation on unseen plots does not.

In [ ]:
null = make_synthetic_table(n_plots=12, n_control=2, n_spectra=60, n_bands=48, effect=0.0, plot_sd=0.3, seed=7)
null = preprocess_table(null).subsample(40)
print("random split by spectrum (INVALID):", round(random_split_demo(null, n_estimators=30), 2))
res = run_classification(null, n_estimators=30)
print("whole-plot validation: plot AUC", round(res.statistic, 2), " permutation p", round(res.p_value, 2))

## 4. Group separation with a real effect
Two complementary plot-level tests: can a model recognise the group of an unseen plot (classifier), and do the groups differ in mean spectrum (PERMANOVA)?

In [ ]:
res = run_classification(clean, n_estimators=30)
print(res.summary())
print(res.plot_scores)
print(group_difference_test(clean).summary())

## 5. What could this design detect?
Inject a known multiplicative change in the NIR bands of the treated plots and see when the test notices it. Report this next to any null result.

In [ ]:
# a table with NO built-in effect: how large an injected change does the design detect?
flat = preprocess_table(make_synthetic_table(n_plots=12, n_control=2, n_spectra=60, n_bands=48, effect=0.0, plot_sd=0.05, seed=1))
detectability(flat, effects=[0, 0.05, 0.1, 0.2, 0.3, 0.5])